[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astheeggeggs/testing_colab/blob/main/LDSC_Practical_2_Colab_flourishes.ipynb)

# 🧬 Practical 2: Genetic Correlation estimation using bivariate LD Score Regression (LDSC)

Welcome to the second practical! Here, we will learn how to estimate the genetic correlation ($r_g$) between two traits using bivariate LD Score Regression (LDSC).


---
## 🛠️ 1. Environment Setup & Data Download

This Colab notebook uses a **Python runtime**, but since the LDSC tools we are using are built in R, we will execute our code using **R cell magic** (`%%R`). We first need to load this extension, download the data, and install the required packages.

Packages used in this session:
* `data.table` & `dplyr`: For fast and efficient data manipulation.
* `remotes`: To install packages directly from GitHub.
* `corrplot`: For visualizing genetic correlation matrices.
* `GenomicSEM`: An R package that includes functions to run LDSC.


In [ ]:
# System dependencies that are often useful for R package installation
!apt-get update -qq
!apt-get install -y -qq libcurl4-openssl-dev libxml2-dev libssl-dev


In [ ]:
# Load the R magic extension to run R code in this Python notebook
%load_ext rpy2.ipython


In [ ]:
# Download the data for this practical
!gdown --folder https://drive.google.com/drive/folders/1DtztFjNBVYP6o72GCIwnUo7MyGjfURtx -O /content/LDSC_Practical_2
!tar -xzvf /content/LDSC_Practical_2/rg.tar.gz -C /content/LDSC_Practical_2
!rm /content/LDSC_Practical_2/rg.tar.gz


In [ ]:
%%R
# Install CRAN packages
install.packages(
  c("data.table", "dplyr", "remotes", "corrplot"),
  repos = "https://cloud.r-project.org",
  quiet=TRUE
)

# Install GenomicSEM from GitHub
remotes::install_github("GenomicSEM/GenomicSEM", upgrade = "never", quiet=TRUE)


### ✅ Load Required Packages

If the installation cell ran successfully, let's load our libraries!


In [ ]:
%%R
suppressWarnings(suppressPackageStartupMessages({
  library(data.table)
  library(dplyr)
  library(GenomicSEM)
  library(corrplot)
}))


---
## 📁 2. Working Directory and File Helpers

The code below assumes that the practical files are available in `/content/LDSC_Practical_2`.
We will also set up some helper functions to make it easier to peek at the files without writing repetitive code.


In [ ]:
%%R
root_dir <- "/content/LDSC_Practical_2/final"

# Keep paths explicit
eur_dir <- file.path(root_dir, "EUR")
eas_dir <- file.path(root_dir, "EAS")

# Helper functions for cleaner file inspection
count_lines <- function(file_path) {
  system(paste("wc -l <", shQuote(file_path)), intern = TRUE)
}

preview_text <- function(file_path, n = 5) {
  cat(paste("--- Preview of", basename(file_path), "---\n"))
  cat(system(paste("head -n", n, shQuote(file_path)), intern = TRUE), sep = "\n")
  cat("\n")
}

preview_gz <- function(file_path, n = 5) {
  cat(paste("--- Preview of", basename(file_path), "---\n"))
  cat(system(paste("zcat", shQuote(file_path), "| head -n", n), intern = TRUE), sep = "\n")
  cat("\n")
}


In this session, we will perform three sets of bivariate linkage disequilibrium score regression (LDSC) analyses.


---
## 🌍 Part 1: Continuous Traits Example in European Genetic Ancestry

This section uses chromosome 1 summary statistics for **BMI** and **Height** in individuals of European (EUR) ancestry.

### 📌 HapMap 3 SNPs
As before, we restrict the analyses to high-quality SNPs in the GWAS summary statistics that overlap with the pre-computed LD scores.


In [ ]:
%%R
hm3_eur <- file.path(eur_dir, "eur_w_ld_chr", "w_hm3.snplist")

# Take a look at the first few lines
preview_text(hm3_eur)

# Count the number of lines
cat("Number of lines in HM3 file:", count_lines(hm3_eur), "\n")


### ❓ Question 1: HM3 SNP Count
How many SNPs are there in the HapMap3 reference file?

<details>
<summary>💡 Click here to reveal the answer!</summary>

There are **1,217,311** SNPs included in the HM3 reference file.

We will restrict the LDSC analyses to the HapMap3 SNPs in this practical because we have the pre-computed LD scores for these SNPs only.
</details>


### 🧹 Step 1: Munge the Data
Let's look at the summary statistics files and their columns before munging.


In [ ]:
%%R
# Look at the summary statistics files and their columns
preview_text(file.path(eur_dir, "GIANT_UKB_BMI_EUR_chr1.txt"))
preview_text(file.path(eur_dir, "Yengo_Height_EUR_chr1.txt"))


### ❓ Question 2: N Column
Are sample sizes ($N$) provided in both sumstats?

<details>
<summary>💡 Click here to reveal the answer!</summary>

**YES!** The columns are conveniently named `N` in both files. This means we do not need to specify the sample size manually during the `munge()` step.
</details>


### ❓ Question 3: Effect Allele
Which column is interpreted as the effect allele (A1) for Height?

<details>
<summary>💡 Click here to reveal the answer!</summary>

The **`EFFECT_ALLELE`** column. `GenomicSEM`'s munge function is smart enough to identify standard column names like this automatically!
</details>


Let's munge the data for both traits. Since we are running bivariate LDSC, we can pass a vector of files to `munge()`.


In [ ]:
%%R
# For multivariate LDSC, we can pass a vector of files to munge()
files_eur <- c(
  file.path(eur_dir, "GIANT_UKB_BMI_EUR_chr1.txt"),
  file.path(eur_dir, "Yengo_Height_EUR_chr1.txt")
)

names_eur <- c(
  "BMI_chr1",
  "Height_chr1"
)

munge(
  files = files_eur,
  hm3 = hm3_eur,
  trait.names = names_eur,
  info.filter = 0.9,
  maf.filter = 0.01
)


Look closely at the munge log printed above to answer the following:

### ❓ Question 4: Munging Details for Height
* How many SNPs were present in the raw sumstats file for Height?
* How many SNPs were dropped because they are not present in the reference HapMap3 file?
* How many SNPs were dropped due to low imputation quality (i.e., INFO)?
* How many SNPs were dropped due to low or missing MAF?
* How many SNPs are left in the munged sumstats file?

<details>
<summary>💡 Click here to reveal the answers!</summary>

* **Raw SNPs:** There were 96,851 SNPs present in the raw sumstats file.
* **Dropped (Not in HM3):** 7,910 SNPs were dropped.
* **Dropped (INFO):** None, as there was no INFO column in the raw dataset.
* **Dropped (MAF):** 0 SNPs were dropped. Usually, there will be few HapMap3 SNPs with MAF < 0.01 - which is why no SNP was dropped here.
* **Final SNPs:** 88,941 SNPs are left in the munged sumstats file.
</details>


### 🚀 Step 2: Run LDSC
Now we are done munging the sumstats. Using the code below, perform LDSC analyses on the munged sumstats.


In [ ]:
%%R
# Examine the munged summary statistics
preview_gz(file.path(eur_dir, "BMI.sumstats.gz"))
preview_gz(file.path(eur_dir, "Height.sumstats.gz"))

traits_eur <- c(
  file.path(eur_dir, "BMI.sumstats.gz"),
  file.path(eur_dir, "Height.sumstats.gz")
)

sample_prev_eur     <- c(NA, NA)
population_prev_eur <- c(NA, NA)

LDSC_EUR <- ldsc(
  traits = traits_eur,
  sample.prev = sample_prev_eur,
  population.prev = population_prev_eur,
  ld = file.path(eur_dir, "eur_w_ld_chr"),
  wld = file.path(eur_dir, "eur_w_ld_chr"),
  trait.names = c("BMI", "Height")
)

# Save the output for later
save(LDSC_EUR, file = file.path(eur_dir, "LDSC_EUR.RData"))


### ❓ Question 5: Prevalence Rates
Why did we use `NA` for sample and population prevalence?

<details>
<summary>💡 Click here to reveal the answer!</summary>

We are examining **continuous traits** here, so prevalence rates are not applicable! They are only used for binary (case/control) traits to convert the heritability to the liability scale.
</details>


### 📊 Inspect the Outputs

The log/output of bivariate LDSC has three sections:
1. Univariate LDSC for Trait 1
2. Univariate LDSC for Trait 2
3. Cross-trait LDSC

### ❓ Question 6: Intercepts and Shared Confounding
* Do the univariate intercepts suggest significant bias/inflation?
* Is there evidence of shared confounding or sample overlap between BMI and height?
* What is the genetic correlation ($r_g$) between BMI and height in the European ancestry sample?

<details>
<summary>💡 Click here to reveal the answers!</summary>

* **Univariate Intercepts:** No. Neither trait has LDSC intercepts significantly greater than 1. (BMI: 0.9355, SE = 0.0803 | Height: 1.3365, SE = 0.3003)
* **Shared Confounding / Overlap:** No. The cross-trait LDSC intercept is not significantly greater than 0 (-0.0688, SE = 0.0772), suggesting no significant shared confounding (like population stratification) or sample overlap.
* **Genetic Correlation:** -0.0272 (SE = 0.0436).
</details>


You can also directly access the matrices generated by `GenomicSEM`.


In [ ]:
%%R
# Genetic covariance matrix
cat("Genetic Covariance Matrix (S):\n")
print(LDSC_EUR$S)

# Genetic correlation matrix
cat("\nGenetic Correlation Matrix:\n")
print(cov2cor(LDSC_EUR$S))

# Standard errors from the V matrix
k <- nrow(LDSC_EUR$S)
SE <- matrix(0, k, k)
SE[lower.tri(SE, diag = TRUE)] <- sqrt(diag(LDSC_EUR$V))
SE[upper.tri(SE)] <- t(SE)[upper.tri(SE)]
cat("\nStandard Errors:\n")
print(SE)


### 🎨 Optional: Create an $r_g$ heatmap
This provides you sample code for creating a correlation heatmap, which is highly useful for manuscripts!


In [ ]:
%%R
rownames(LDSC_EUR$S) <- colnames(LDSC_EUR$S)

corrplot(
  corr = cov2cor(LDSC_EUR$S),
  method = "color",
  addCoef.col = "dark grey",
  add = FALSE,
  bg = "white",
  diag = TRUE,
  outline = TRUE,
  tl.col = "black",
  mar = c(1,1,2,1),
  title = "Genetic Correlation: BMI & Height (EUR)"
)


---
## 🌏 Part 2: Continuous Traits Example in East Asian Ancestry

The summary statistics and LD scores for this section are from East Asian (EAS) ancestry. We will perform similar bivariate LDSC analyses for BMI and Height.

As before, we start by examining if the sumstats files have the necessary columns.


In [ ]:
%%R
# Look at the files
preview_text(file.path(eas_dir, "BMI_EAS_chr1.txt"))
preview_text(file.path(eas_dir, "Yengo_Height_EAS_chr1.txt"))


### ❓ Question 7: Identifying Columns
* Is the sample size ($N$) column present for both?
* Which column is the effect allele (A1) for each trait?

<details>
<summary>💡 Click here to reveal the answers!</summary>

* **Sample Size (N):** The BMI sumstats *do not* include N! So, we have to manually specify `N` for BMI when we run `munge()`.
* **Effect Allele (A1):** The effect allele is `ALLELE1` in the BMI sumstats and `EFFECT_ALLELE` in the Height sumstats.
</details>


In [ ]:
%%R
hm3_eas <- file.path(eas_dir, "eas_ldscores", "w_hm3.snplist")
preview_text(hm3_eas)
cat("Number of lines in EAS HM3 file:", count_lines(hm3_eas), "\n")

files_eas <- c(
  file.path(eas_dir, "BMI_EAS_chr1.txt"),
  file.path(eas_dir, "Yengo_Height_EAS_chr1.txt")
)

names_eas <- c("BMI_EAS_chr1", "Height_EAS_chr1")

# Note that we are using ancestrally matched LD scores!
# We also provide a vector for N. The first is for BMI (since it was missing),
# and the second is NA because Height already has an N column.
munge(
  files = files_eas,
  hm3 = hm3_eas,
  trait.names = names_eas,
  N = c(163835, NA),
  info.filter = 0.9,
  maf.filter = 0.01
)


Now, we can run LDSC analyses as before.


In [ ]:
%%R
preview_gz(file.path(eas_dir, "BMI.sumstats.gz"))
preview_gz(file.path(eas_dir, "Height.sumstats.gz"))

traits_eas <- c(
  file.path(eas_dir, "BMI.sumstats.gz"),
  file.path(eas_dir, "Height.sumstats.gz")
)

sample_prev_eas     <- c(NA, NA)
population_prev_eas <- c(NA, NA)

LDSC_EAS <- ldsc(
  traits = traits_eas,
  sample.prev = sample_prev_eas,
  population.prev = population_prev_eas,
  ld = file.path(eas_dir, "eas_ldscores"),
  wld = file.path(eas_dir, "eas_ldscores"),
  trait.names = c("BMI", "Height")
)


### ❓ Question 8: Comparing Heritabilities
How do the $h^2_{\text{SNP}}$ estimates for BMI and Height in East Asians compare to the European estimates from Part 1?

<details>
<summary>💡 Click here to reveal the answer!</summary>

The $h^2_{\text{SNP}}$ estimates of both BMI and Height are lower in the East Asian ancestry sample than the European ancestry sample:
* **BMI:** `0.14` in EAS vs `0.21` in EUR
* **Height:** `0.24` in EAS vs `0.42` in EUR

*We might discuss later why there is a difference! (Hint: differences in sample size, imputation quality, LD reference panels, or true genetic architecture).*
</details>


### ❓ Question 9: EAS Genetic Correlation
What is the genetic correlation between BMI and Height in the EAS ancestry sample?

<details>
<summary>💡 Click here to reveal the answer!</summary>

The genetic correlation between BMI and height in the EAS ancestry sample is **`-0.129`** (SE = `0.077`).
</details>


In [ ]:
%%R
rownames(LDSC_EAS$S) <- colnames(LDSC_EAS$S)

corrplot(
  corr = cov2cor(LDSC_EAS$S),
  method = "color",
  addCoef.col = "dark grey",
  add = FALSE,
  bg = "white",
  diag = TRUE,
  outline = TRUE,
  tl.col = "black",
  mar = c(1,1,2,1),
  title = "Genetic Correlation: BMI & Height (EAS)"
)


---
## 🧠 Part 3: Binary (Case/Control) Traits Example in European Ancestry

Now, we will perform bivariate LDSC analyses for two psychiatric disorders: **Schizophrenia (SCZ)** and **Bipolar Disorder (BIP)** in European ancestry.

As before, we will first examine the columns in the GWAS sumstats file.


In [ ]:
%%R
# Files
preview_text(file.path(eur_dir, "SCZ_chr1.txt"))
preview_text(file.path(eur_dir, "BIP_chr1.txt"))


### ❓ Question 10: Effective Sample Size
How are the sample sizes reported in the SCZ and BIP summary statistics?

<details>
<summary>💡 Click here to reveal the answer!</summary>

* SCZ sumstats include the effective $N$ (`NEFF`).
* BIP sumstats include `NEFFDIV2`, which is half of the effective $N$.

Note that `munge()` can usually recognize these different types of $N$ and interpret them properly!
</details>


In [ ]:
%%R
files_bin <- c(
  file.path(eur_dir, "SCZ_chr1.txt"),
  file.path(eur_dir, "BIP_chr1.txt")
)

names_bin <- c("SCZ","BIP")

munge(
  files = files_bin,
  hm3 = hm3_eur,
  trait.names = names_bin,
  info.filter = 0.9,
  maf.filter = 0.01
)


### ❓ Question 11: Munge $N$ interpretation
Look at the munge log. How was $N$ calculated for the BIP sumstats?

<details>
<summary>💡 Click here to reveal the answer!</summary>

For BIP, $N_{\text{eff}}$ is calculated by doubling `NEFFDIV2`.
If `munge()` does not identify the column correctly or doesn't double `NEFFDIV2`, you would need to manually create a new column (`NEFF = 2 * NEFFDIV2`) in the raw dataset before running `munge()`.
</details>


Note that we need to specify the sample prevalence and population prevalence to obtain liability-scale estimates for binary traits.

The population prevalences of SCZ and BIP (0.01 and 0.02, respectively) are obtained from published epidemiological studies.

### ❓ Question 12: Sample prevalence for effective $N$
What is the "sample prevalence" when we use the effective $N$ (rather than the actual $N$, which is the sum of $N_{\text{cases}}$ and $N_{\text{controls}}$)?

<details>
<summary>💡 Click here to reveal the answer!</summary>
Effective $N$ implies sample prevalence = 0.5.
</details>

Now, we run `ldsc()`.
LDSC will first report the **observed-scale** estimates, then the **liability-scale** estimates.


In [ ]:
%%R
preview_gz(file.path(eur_dir, "SCZ.sumstats.gz"))
preview_gz(file.path(eur_dir, "BIP.sumstats.gz"))

traits_bin <- c(
  file.path(eur_dir, "SCZ.sumstats.gz"),
  file.path(eur_dir, "BIP.sumstats.gz")
)

# For binary traits we must specify sample and population prevalence!
# Because the sumstats use Effective N, we set sample.prev = 0.5 (meaning 50% cases / 50% controls)
sample_prev_bin     <- c(0.5, 0.5)
population_prev_bin <- c(0.01, 0.02) # SCZ ~ 1%, BIP ~ 2%

LDSC_SCZ_BIP <- ldsc(
  traits = traits_bin,
  sample.prev = sample_prev_bin,
  population.prev = population_prev_bin,
  ld = file.path(eur_dir, "eur_w_ld_chr"),
  wld = file.path(eur_dir, "eur_w_ld_chr"),
  trait.names = c("SCZ", "BIP")
)


### ❓ Question 13: Intercepts and Inflation
What are the univariate LDSC intercepts for SCZ and BIP? Do they suggest bias?

<details>
<summary>💡 Click here to reveal the answer!</summary>

* **SCZ:** LDSC intercept = 1.1072 (SE = 0.0523). Attenuation Ratio = 0.0896 (SE = 0.0437)
* **BIP:** LDSC intercept = 1.0032 (SE = 0.0349). Attenuation Ratio = 0.005 (SE = 0.055)

Though SCZ suggests a small amount of bias (~8% of the inflation is not due to polygenicity), the vast majority of the signal for both traits is highly polygenic.
</details>


### ❓ Question 14: Shared Confounding
What is the cross-trait LDSC intercept? What does this mean?

<details>
<summary>💡 Click here to reveal the answer!</summary>

Cross-trait LDSC intercept = 0.1945 (SE = 0.0315).

Since this is significantly greater than 0, it suggests likely **sample overlap** (e.g., the same controls were used for both the SCZ and BIP GWAS).
There could also be some shared residual population stratification.

Note that the genetic covariance and correlation are estimated after accounting for these factors (sample overlap or shared confounding).
</details>


### ❓ Question 15: Genetic Correlation
What are the genetic covariance and genetic correlation between SCZ and BIP?

<details>
<summary>💡 Click here to reveal the answer!</summary>

* **Genetic covariance (liability-scale)** = 0.1511 (SE = 0.0168)
* **Genetic correlation ($r_g$)** = 0.7276 (SE = 0.0809)

This is a massive genetic correlation, indicating strong shared genetic architecture between Schizophrenia and Bipolar Disorder!
</details>


In [ ]:
%%R
rownames(LDSC_SCZ_BIP$S) <- colnames(LDSC_SCZ_BIP$S)

corrplot(
  corr = cov2cor(LDSC_SCZ_BIP$S),
  method = "color",
  addCoef.col = "dark grey",
  add = FALSE,
  bg = "white",
  diag = TRUE,
  outline = TRUE,
  tl.col = "black",
  mar = c(1,1,2,1),
  title = "Genetic Correlation: SCZ & BIP (EUR)"
)


---
## 🏁 End of Practical 2

You have now completed Practical 2 for estimating Genetic Correlation using LDSC!
